# Участник 1: Linear + Huber (два таргета)

В этом ноутбуке делаем базовый линейный блок для участника 1.
Сравниваем Huber-подходы на двух таргетах:
- target_next_session_length_sec
- future_sessions_mean_playtime_7d

Стратегии train_capped_target, train_quantile_04, train_engagement_quantile_035 тут не используются
Используем только трансформации p995 и log1p_p995

In [15]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

repo_root = Path.cwd()
if repo_root.name != 'ml_in_gamedev_project':
    for pp in [repo_root, *repo_root.parents]:
        if pp.name == 'ml_in_gamedev_project':
            repo_root = pp
            break
sys.path.append(str(repo_root))

from preprocessing.preprocessing import (
    load_data,
    prepare_for_targets,
    regression_metrics,
    TargetTransform,
)

warnings.filterwarnings('ignore')


In [16]:
df = load_data()
df.shape

(3438527, 84)

In [17]:
df.head(2)

,appmetrica_device_id,installation_id,session_id,start,end,duration_seconds,duration_hms,events_count,target_next_session_length_sec,past_sessions_count,...,events_span_sec,first_event_timestamp,last_event_timestamp,unique_device_models,unique_device_types,unique_app_versions,unique_countries,unique_cities,unique_connection_types,target_log1p
0,2.393465e+13,a8950b8e026a4f1d971615f7db5e641d,10000000002,2026-04-05 23:20:27,2026-04-05 23:20:28,1,00:00:01,26,3.0,0,...,1.0,1775420427,1775420428,1,1,1,1,0,1,1.386294
1,8.698994e+13,ec6ec4a527eb4ebe838a1b341abd567d,10000000001,2025-12-29 23:10:22,2025-12-29 23:10:29,7,00:00:07,67,39.0,0,...,7.0,1767039022,1767039029,1,1,1,1,0,1,3.688879


## Подготовка двух таргетов

Берем два таргета сразу

In [18]:
targets = ['target_next_session_length_sec', 'future_sessions_mean_playtime_7d']
packs = prepare_for_targets(df, target_cols=targets, max_rows=50000)

sizes = []
for t, p in packs.items():
    sizes.append({
        'target': t,
        'train_rows': len(p.x_train),
        'val_rows': len(p.x_val),
        'test_rows': len(p.x_test),
        'features': len(p.feature_cols),
    })
pd.DataFrame(sizes)

,target,train_rows,val_rows,test_rows,features
0,target_next_session_length_sec,35000,7500,7500,73
1,future_sessions_mean_playtime_7d,35000,7500,7500,73


## План эксперимента

Пробуем 2 модели (HuberRegressor, SGDRegressor(loss='huber')) и 2 трансформации таргета (p995, log1p_p995)

По валидации выбираем лучшую комбинацию,
потом считаем тестовые метрики 

In [19]:
def make_preprocessor(num_cols, cat_cols):
    num_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, max_categories=30, sparse_output=False)),
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols),
    ], remainder='drop', verbose_feature_names_out=False)


In [20]:
models = {
    'HuberRegressor': HuberRegressor(epsilon=1.35, alpha=0.0002, max_iter=300),
    'SGDHuber': SGDRegressor(loss='huber', epsilon=0.1, alpha=0.0005, penalty='l2', max_iter=1000, random_state=42),
}
modes = ['p995', 'log1p_p995']

In [21]:
rows = []

for t, p in packs.items():
    prep = make_preprocessor(p.num_cols, p.cat_cols)
    xtr = prep.fit_transform(p.x_train)
    xva = prep.transform(p.x_val)
    xte = prep.transform(p.x_test)

    y_tr_raw = p.y_train
    y_va_raw = p.y_val
    y_te_raw = p.y_test

    for mode in modes:
        tfm = TargetTransform(mode=mode).fit(y_tr_raw)
        ytr = tfm.transform(y_tr_raw)

        for m_name, base in models.items():
            mdl = clone(base)
            mdl.fit(xtr, ytr)

            ptr = tfm.inverse(mdl.predict(xtr))
            pva = tfm.inverse(mdl.predict(xva))
            pte = tfm.inverse(mdl.predict(xte))

            train_m = regression_metrics(y_tr_raw, ptr)
            val_m = regression_metrics(y_va_raw, pva)
            test_m = regression_metrics(y_te_raw, pte)

            r = {
                'target': t,
                'model': m_name,
                'target_mode': mode,
            }
            for k, v in train_m.items():
                r[f'train_{k}'] = v
            for k, v in val_m.items():
                r[f'val_{k}'] = v
            for k, v in test_m.items():
                r[f'test_{k}'] = v
            rows.append(r)

res = pd.DataFrame(rows).sort_values(['target', 'val_mae']).reset_index(drop=True)
res

,target,model,target_mode,train_mae,train_medae,train_p70_abs_error,train_p90_abs_error,train_r2,train_product_mae,train_engagement_risk_mae,...,test_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae
0,future_sessions_mean_playtime_7d,HuberRegressor,p995,313.371761,192.013172,326.200683,678.750272,0.179000,221.985160,236.545608,...,308.328872,199.266054,333.545229,672.602828,0.149954,230.766091,245.235878,292.636019,200.229606,1008.157126
1,future_sessions_mean_playtime_7d,SGDHuber,p995,351.271811,202.832938,366.979947,801.820097,-0.040370,219.118985,232.851280,...,336.165149,207.372799,355.115161,764.962997,-0.047522,224.105856,235.952837,224.718961,231.122924,1308.021771
2,future_sessions_mean_playtime_7d,HuberRegressor,log1p_p995,350.662917,187.381143,327.901959,712.958372,-9.140087,215.147329,259.252498,...,342.241523,195.759942,332.040840,705.434299,-0.719249,223.328908,246.967330,266.436069,219.148892,1314.888107
3,future_sessions_mean_playtime_7d,SGDHuber,log1p_p995,355.275273,191.926253,331.175727,714.482896,-10.334101,218.264236,266.326244,...,336.388699,195.911765,329.939052,697.221921,-0.603019,226.980470,250.077300,273.680986,220.279356,1227.168474
4,target_next_session_length_sec,HuberRegressor,p995,545.010407,271.077249,434.463255,1196.693734,0.031211,269.661808,285.523732,...,479.459086,255.444750,394.908726,1068.284711,0.032907,260.729651,271.674438,249.372736,276.338231,1885.483079
5,target_next_session_length_sec,SGDHuber,p995,560.539624,202.469584,417.985297,1391.947365,-0.057101,223.654158,236.461036,...,482.171271,189.614009,340.947028,1238.775290,-0.106591,212.278046,222.080410,131.571715,383.755008,2110.222357
6,target_next_session_length_sec,HuberRegressor,log1p_p995,569.377271,224.774020,435.046941,1305.075127,-0.876873,241.555436,283.461830,...,505.835211,204.724013,384.763142,1180.532996,-4.088542,230.886928,247.434202,180.139052,351.727977,2170.992070
7,target_next_session_length_sec,SGDHuber,log1p_p995,574.276586,235.184763,439.086629,1300.355850,-1.148506,248.941507,294.431650,...,506.272630,217.982654,391.203753,1152.511735,-4.209586,239.540915,254.393994,196.622184,336.574147,2145.889484


## Интерпретация

Смотрим не только на mae, но и на product_mae и engagement_risk_mae
Это даст более честный baseline для продуктовых экспериментов в следующих ноутбуках

In [22]:
main_cols = [
    'target', 'model', 'target_mode',
    'train_mae', 'val_mae', 'test_mae',
    'train_product_mae', 'val_product_mae', 'test_product_mae',
    'train_engagement_risk_mae', 'val_engagement_risk_mae', 'test_engagement_risk_mae',
    'test_medae', 'test_p70_abs_error', 'test_p90_abs_error', 'test_r2',
    'test_small_mae', 'test_normal_mae', 'test_long_mae',
]
res[main_cols]


,target,model,target_mode,train_mae,val_mae,test_mae,train_product_mae,val_product_mae,test_product_mae,train_engagement_risk_mae,val_engagement_risk_mae,test_engagement_risk_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_small_mae,test_normal_mae,test_long_mae
0,future_sessions_mean_playtime_7d,HuberRegressor,p995,313.371761,313.668615,308.328872,221.985160,219.531531,230.766091,236.545608,232.483834,245.235878,199.266054,333.545229,672.602828,0.149954,292.636019,200.229606,1008.157126
1,future_sessions_mean_playtime_7d,SGDHuber,p995,351.271811,354.558977,336.165149,219.118985,225.747034,224.105856,232.851280,238.657822,235.952837,207.372799,355.115161,764.962997,-0.047522,224.718961,231.122924,1308.021771
2,future_sessions_mean_playtime_7d,HuberRegressor,log1p_p995,350.662917,356.827036,342.241523,215.147329,215.596221,223.328908,259.252498,243.793207,246.967330,195.759942,332.040840,705.434299,-0.719249,266.436069,219.148892,1314.888107
3,future_sessions_mean_playtime_7d,SGDHuber,log1p_p995,355.275273,364.425823,336.388699,218.264236,218.255040,226.980470,266.326244,250.417972,250.077300,195.911765,329.939052,697.221921,-0.603019,273.680986,220.279356,1227.168474
4,target_next_session_length_sec,HuberRegressor,p995,545.010407,533.726747,479.459086,269.661808,273.585905,260.729651,285.523732,288.043930,271.674438,255.444750,394.908726,1068.284711,0.032907,249.372736,276.338231,1885.483079
5,target_next_session_length_sec,SGDHuber,p995,560.539624,544.569094,482.171271,223.654158,224.101568,212.278046,236.461036,236.704958,222.080410,189.614009,340.947028,1238.775290,-0.106591,131.571715,383.755008,2110.222357
6,target_next_session_length_sec,HuberRegressor,log1p_p995,569.377271,576.515733,505.835211,241.555436,243.820311,230.886928,283.461830,279.127238,247.434202,204.724013,384.763142,1180.532996,-4.088542,180.139052,351.727977,2170.992070
7,target_next_session_length_sec,SGDHuber,log1p_p995,574.276586,582.752265,506.272630,248.941507,254.854673,239.540915,294.431650,290.624772,254.393994,217.982654,391.203753,1152.511735,-4.209586,196.622184,336.574147,2145.889484


In [23]:
best_mae = res.sort_values('test_mae').groupby('target', as_index=False).first()
best_prod = res.sort_values('test_product_mae').groupby('target', as_index=False).first()
best_eng = res.sort_values('test_engagement_risk_mae').groupby('target', as_index=False).first()

best_mae[['target','model','target_mode','train_mae','val_mae','test_mae','test_product_mae','test_engagement_risk_mae']]


,target,model,target_mode,train_mae,val_mae,test_mae,test_product_mae,test_engagement_risk_mae
0,future_sessions_mean_playtime_7d,HuberRegressor,p995,313.371761,313.668615,308.328872,230.766091,245.235878
1,target_next_session_length_sec,HuberRegressor,p995,545.010407,533.726747,479.459086,260.729651,271.674438


In [24]:
best_prod[['target','model','target_mode','train_mae','val_mae','test_mae','test_product_mae','test_engagement_risk_mae']]


,target,model,target_mode,train_mae,val_mae,test_mae,test_product_mae,test_engagement_risk_mae
0,future_sessions_mean_playtime_7d,HuberRegressor,log1p_p995,350.662917,356.827036,342.241523,223.328908,246.96733
1,target_next_session_length_sec,SGDHuber,p995,560.539624,544.569094,482.171271,212.278046,222.08041


In [25]:
best_eng[['target','model','target_mode','train_mae','val_mae','test_mae','test_product_mae','test_engagement_risk_mae']]


,target,model,target_mode,train_mae,val_mae,test_mae,test_product_mae,test_engagement_risk_mae
0,future_sessions_mean_playtime_7d,SGDHuber,p995,351.271811,354.558977,336.165149,224.105856,235.952837
1,target_next_session_length_sec,SGDHuber,p995,560.539624,544.569094,482.171271,212.278046,222.080410


## Финальный вывод 


По результатам линейного baseline на двух таргетах видно, что для `future_sessions_mean_playtime_7d` лучший по обычному качеству (test MAE) вариант — `HuberRegressor + p995` (самый низкий MAE и при этом адекватные product/engagement метрики), но если оптимизироваться именно под продуктовую ошибку, то `HuberRegressor + log1p_p995` даёт лучший `test_product_mae` ценой ухудшения MAE, а по риску для engagement (`test_engagement_risk_mae`) чуть лучше выглядит `SGDHuber + p995`. Для `target_next_session_length_sec` минимальный test MAE снова у `HuberRegressor + p995`, но разница с `SGDHuber + p995` по MAE небольшая, зато `SGDHuber + p995` заметно выигрывает по `test_product_mae` и `test_engagement_risk_mae`, поэтому как продуктовый бейзлайн для следующей части экспериментов логично держать именно его, а `HuberRegressor + p995` - как MAE-ориентированную точку отсчёта.